In [2]:
#import socket
#import pickle

#soc = socket.socket()
#hostname = "localhost"  # 127.0.0.1 #0.0.0.0
#port = 5000
#soc.bind((hostname, port))
#soc.listen(5)
#conn, addr = soc.accept()
#print("Device Connected")

#while True:
#        msg = b"Hello World"
#        msg = pickle.dumps(input("Enter the message : "))
#        conn.send(msg)
#        if msg == pickle.dumps("exit"):
#            break

In [3]:
#import socket

#mySocket = socket.socket()
#mySocket.bind(("localhost", 5000))
#mySocket.listen(5)

#print("Waiting for C# game...")
#conn, addr = mySocket.accept()
#print("Device Connected")

#msg = bytes("Hello from Python", "utf-8")
#conn.send(msg)

#conn.close()
#mySocket.close()

In [ ]:
import asyncio
import csv
import socket
import cv2
import mediapipe as mp
import time
from dollarpy import Recognizer, Template, Point
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic


def load_users():
    users = []
    try:
        with open("users.csv", "r", encoding="utf-8") as file:
            reader = csv.reader(file)
            next(reader)

            for row in reader:
                if len(row) >= 3:
                    name = row[0].strip()
                    bt_id = row[1].strip().lower()
                    gender = row[2].strip().lower()
                    users.append((name, bt_id, gender))
    except FileNotFoundError:
        print("Error: users.csv not found.")
    except Exception as e:
        print("Error while reading users.csv:", str(e))

    return users


def getPoints(videoURL, label):
    cap = cv2.VideoCapture(videoURL)

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        points = []
        stroke_id = 1

        while cap.isOpened():
            ret, frame = cap.read()

            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False

            results = holistic.process(image)

            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(
                    image,
                    results.right_hand_landmarks,
                    mp_holistic.HAND_CONNECTIONS
                )

                index_tip = results.right_hand_landmarks.landmark[8]

                h, w, _ = image.shape
                x = int(index_tip.x * w)
                y = int(index_tip.y * h)

                points.append(Point(x, y, stroke_id))
                cv2.circle(image, (x, y), 8, (0, 255, 0), -1)

        cap.release()
        cv2.destroyAllWindows()
        return points


templates = []


def add_template(label, video_path):
    points = getPoints(video_path, label)
    print(label, video_path, "points =", len(points))

    if len(points) > 0:
        templates.append(Template(label, points))
    else:
        print("Skipped empty template:", video_path)


add_template("circle", " circle/circle_1.mp4")
add_template("circle", " circle/circle_2.mp4")

add_template("slide_left", " slide_left/slide_left_1.mp4")
add_template("slide_left", " slide_left/slide_left_2.mp4")

add_template("slide_right", " slide_right/slide_right_1.mp4")
add_template("slide_right", " slide_right/slide_right_2.mp4")

add_template("fist", " fist/fist_1.mp4")
add_template("fist", " fist/fist_2.mp4")

print("Templates loaded:", len(templates))

if len(templates) == 0:
    raise Exception("No valid templates were loaded.")

recognizer = Recognizer(templates)


def send_message(conn, msg):
    try:
        conn.send((msg + "\n").encode("utf-8"))
        print("Sent:", msg)
    except Exception as e:
        print("Send Error:", str(e))


async def scan_and_login(conn):
    users = load_users()

    if not users:
        print("No users found in CSV. Skipping Bluetooth login.")
        return False

    print("Users from CSV:")
    for name, bt_id, gender in users:
        print("CSV ->", name, bt_id, gender)

    print("Scanning for Bluetooth devices...")

    def run_scan():
        async def _scan():
            from bleak import BleakScanner
            return await BleakScanner.discover(timeout=20.0)
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(_scan())
        finally:
            loop.close()

    try:
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as pool:
            devices = pool.submit(run_scan).result()

        print("\nDevices found:")
        for device in devices:
            print("FOUND ->", device.name, device.address)

        for device in devices:
            if not device.address:
                continue
            addr = device.address.strip().lower()
            for name, bt_id, gender in users:
                if addr == bt_id:
                    print("MATCH FOUND!")
                    print("User Logged In:", name)
                    print("Gender:", gender)
                    send_message(conn, f"LOGIN:{name}:{gender}")
                    return True

    except Exception as e:
        print("Bluetooth Error:", str(e))

    print("No matching user found.")
    return False


last_label = ""
last_time = 0


def send_gesture(conn, label):
    global last_label, last_time

    now = time.time()

    if label == last_label and now - last_time < 1.0:
        return

    if label == "slide_right":
        msg = "GESTURE:RIGHT"
    elif label == "slide_left":
        msg = "GESTURE:LEFT"
    elif label == "circle":
        msg = "GESTURE:CIRCLE"
    elif label == "fist":
        msg = "GESTURE:FIST"
    else:
        return

    send_message(conn, msg)

    last_label = label
    last_time = now


def recognize_live_gesture(conn):
    cap = cv2.VideoCapture(1)

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        points = []
        stroke_id = 1
        tracking = False

        while cap.isOpened():
            ret, frame = cap.read()

            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False

            results = holistic.process(image)

            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(
                    image,
                    results.right_hand_landmarks,
                    mp_holistic.HAND_CONNECTIONS
                )

                index_tip = results.right_hand_landmarks.landmark[8]

                h, w, _ = image.shape
                x = int(index_tip.x * w)
                y = int(index_tip.y * h)

                points.append(Point(x, y, stroke_id))
                cv2.circle(image, (x, y), 8, (0, 255, 0), -1)
                tracking = True

            else:
                if tracking and len(points) > 10 and len(templates) > 0:
                    try:
                        result = recognizer.recognize(points)

                        if result is not None:
                            label = result[0]
                            score = result[1]

                            print("Predicted:", label)
                            print("Score:", score)

                            send_gesture(conn, label)

                    except Exception as e:
                        print("Recognition error:", str(e))

                points = []
                tracking = False

            cv2.imshow("Live Gesture Recognition", image)

            key = cv2.waitKey(1) & 0xFF
            if key == 27:
                break

        cap.release()
        cv2.destroyAllWindows()


async def main():
    soc = socket.socket()
    soc.bind(("localhost", 5000))
    soc.listen(1)

    print("Waiting for C# game to connect on port 5000...")
    conn, addr = soc.accept()
    print("C# connected:", addr)

    while True:
        success = await scan_and_login(conn)
        if success:
            break
        print("Retrying login in 5 seconds...")
        await asyncio.sleep(5)

    # Now outside the loop — runs after successful login
    recognize_live_gesture(conn)

    conn.close()
    soc.close()
    


await main()

c:\Uni\anaconda\envs\mp_env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: message_factory.GetMessageClass() is deprecated. Please use message_factory.GetMessageClass() instead. message_factory.GetMessageClass() will be removed soon.
  warnings.warn('message_factory.GetMessageClass() is deprecated. Please '


circle  circle/circle_1.mp4 points = 68
circle  circle/circle_2.mp4 points = 105
slide_left  slide_left/slide_left_1.mp4 points = 42
slide_left  slide_left/slide_left_2.mp4 points = 26
slide_right  slide_right/slide_right_1.mp4 points = 11
slide_right  slide_right/slide_right_2.mp4 points = 16
fist  fist/fist_1.mp4 points = 60
fist  fist/fist_2.mp4 points = 51
Templates loaded: 8
Waiting for C# game to connect on port 5000...
C# connected: ('127.0.0.1', 60899)
Users from CSV:
CSV -> Anas 98:50:2e:70:21:3b m
CSV -> Sara dc:91:66:0f:90:e0 f
CSV -> Omar 94:24:b8:4b:06:38 m
CSV -> Yosof 71:55:76:bb:5c:3c m
CSV -> Evil_Yosof 4f:e0:db:ea:ce:d1 f
CSV -> Eviler_Yosof 7d:85:fa:2d:67:a7 f
CSV -> The_Good_Yosof 74:e5:ab:b2:03:44 m
CSV -> Evil_Sarah 75:1c:54:0c:92:3c f
CSV -> Eviler_Sarah 47:85:c3:7b:3b:47 f
CSV -> EVILEST_SARAH 41:02:23:bc:06:ce f
CSV -> HCI_prof 6a:f6:23:70:fe:93 m
CSV -> testing 08:e4:df:ac:8d:8d f
CSV -> testin2 47:fc:d7:8a:f4:a6 f
CSV -> testing3 45:63:ae:a4:36:cd m
CSV -> te

In [22]:
recognize_live_gesture(None)


Predicted: slide_left
Score: 0.41226469183040393
Send Error: 'NoneType' object has no attribute 'send'
Predicted: slide_left
Score: 0.0898264550665695
Send Error: 'NoneType' object has no attribute 'send'


In [1]:
# pip install bleak
import asyncio
import csv
from bleak import BleakScanner
from datetime import datetime


def load_users():
    users = []

    with open("users.csv", "r", encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)  # skip header

        for row in reader:
            name = row[0].strip()
            bt_id = row[1].strip().lower()
            users.append((name, bt_id))

    return users


async def scan_bluetooth_devices():
    print("Scanning for bluetooth devices...")
    scan_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    try:
        users = load_users()
        devices = await BleakScanner.discover()

        print(f"\nScan completed at {scan_time}")
        print(f"Found {len(devices)} devices:\n")

        found_match = False

        for device in devices:
            device_name = device.name or "Unknown"
            device_address = device.address.strip().lower()

            print(f"Device Name: {device_name}")
            print(f"MAC Address: {device_address}")

            for user in users:
                if device_address == user[1]:
                    print("MATCH FOUND!")
                    print("User Logged In:", user[0])
                    found_match = True

        if found_match == False:
            print("No matching user found.")

    except Exception as e:
        print(f"An error occurred: {str(e)}")


async def main():
    print("Starting Bluetooth scan...")
    await scan_bluetooth_devices()


async def main():
    print("Starting Bluetooth scan...")
    await scan_bluetooth_devices()

await main()

Starting Bluetooth scan...
Scanning for bluetooth devices...

Scan completed at 2026-03-24 09:28:07
Found 91 devices:

Device Name: Unknown
MAC Address: 13:40:fc:59:5b:20
Device Name: Unknown
MAC Address: 2d:ff:73:12:12:e5
Device Name: Unknown
MAC Address: 13:26:c8:e5:74:92
Device Name: Unknown
MAC Address: 14:52:10:f7:58:bb
Device Name: Unknown
MAC Address: 07:6a:5e:1e:48:6d
Device Name: Unknown
MAC Address: 77:2c:95:35:10:dc
Device Name: Unknown
MAC Address: 65:32:f8:7c:82:ce
Device Name: OPPO Enco Buds3
MAC Address: 64:8f:db:e8:d8:c9
Device Name: Unknown
MAC Address: 73:c6:db:0b:fa:ed
Device Name: OPPO Enco Buds3
MAC Address: 63:e7:f7:d9:f3:3e
Device Name: Unknown
MAC Address: 4d:20:72:d7:4d:26
Device Name: Unknown
MAC Address: 75:24:86:dd:04:b8
Device Name: Unknown
MAC Address: 77:9b:7d:b6:3f:f0
Device Name: Unknown
MAC Address: 55:ee:31:36:76:72
Device Name: Unknown
MAC Address: 51:9a:89:02:c0:97
Device Name: Unknown
MAC Address: 3c:f5:37:1c:f6:d4
Device Name: Unknown
MAC Address:

In [ ]:
# This is used to close the socket connection after finishing the communication
soc.close()

In [ ]:
!pip install bleak


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
